# PRESAGE 最小样本查看

论文: https://doi.org/10.1101/2025.06.03.657653

归类: 基因扰动响应预测／多来源基因先验融合

展示h5ad结构、扰动标签、基因嵌入、预处理核验和资源复用关系。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

base = Path('..')
print(f'项目目录: {base.resolve()}')

# 加载数据清单
datasets = pd.read_csv(base / 'data_manifest' / 'datasets.csv')
print(f'\n=== 资源清单 ({len(datasets)} 条) ===')
print(datasets[['dataset_id','category','name','download_status']].to_string(index=False))

# 复用资源映射
reuse = pd.read_csv(base / 'data_manifest' / 'reuse_resource_map.csv')
print(f'\n=== 复用资源 ({len(reuse)} 条) ===')
print(reuse[['resource_name','presage_role','existing_module']].to_string(index=False))

## 1. h5ad 文件查看

In [ ]:
h5ad_files = list((base / 'data' / 'raw' / 'presage_cache').glob('*.h5ad'))
if h5ad_files:
    import anndata
    f = h5ad_files[0]
    adata = anndata.read_h5ad(f, backed='r')
    print(f'文件: {f.name}')
    print(f'形状: {adata.shape} (细胞 × 基因)')
    print(f'obs列: {list(adata.obs.columns)}')
    print(f'var列: {list(adata.var.columns)}')
    print(f'layers: {list(adata.layers.keys())}')
    # 扰动标签
    if 'perturbation' in adata.obs.columns:
        print(f'\n扰动数: {adata.obs["perturbation"].nunique()}')
        print(f'前5个扰动:')
        for p, c in adata.obs['perturbation'].value_counts().head(5).items():
            print(f'  {p}: {c} 细胞')
else:
    print('未找到h5ad文件。请先运行 download_presage_cache.py --download')
    print('Zenodo: https://zenodo.org/records/15587986')

## 2. 基因嵌入查看

In [ ]:
emb_files = list((base / 'data' / 'raw').glob('**/*.npy')) + list((base / 'data' / 'raw').glob('**/*emb*.csv'))
if emb_files:
    f = emb_files[0]
    print(f'嵌入文件: {f.name}')
    if f.suffix == '.npy':
        arr = np.load(f, allow_pickle=True)
        if arr.dtype == object:
            d = arr.item()
            genes = list(d.keys())
            print(f'基因数: {len(genes)}')
            print(f'嵌入维度: {d[genes[0]].shape}')
            print(f'前5个基因: {genes[:5]}')
else:
    print('未找到嵌入文件。')
    print('GenePT嵌入可复用 datasets/adapert/ 中的资源。')

## 3. 预处理核验

In [ ]:
print('=== 预期预处理逻辑 ===')
print('1. 归一化对数表达 (log1p CP10K)')
print('2. 减去对照均值')
print('3. 按扰动求平均')
print()
print('=== 基因数核验 ===')
print('HVG之外可能补回被扰动基因，输出不一定只有5,000列。')
print()
print('运行 scripts/verify_preprocessing.py --file <h5ad> 可自动核验。')

## 4. 数据划分与防止泄漏

In [ ]:
print('=== 数据划分原则 ===')
print('1. 保留官方划分')
print('2. 目标系统测试扰动的真实响应不加入训练')
print('3. 同一基因在其他允许使用的数据集中出现，记为外部先验')
print('4. 跨数据集先验的使用关系需记录')
print()
print('=== 复用已有资源 ===')
print('Replogle: datasets/gears/, datasets/xcell/')
print('scPerturb: datasets/scperturb/')
print('DepMap: datasets/xcell/')
print('GenePT: datasets/adapert/')
print('STRING: datasets/adapert/')